# 06 · Monitoring and verdicts

*What happens to a verdict after it's issued, and who watches the checks.*

Notebook 05 ran the checks once a month on the planted data, the way a scheduled job would, and recorded a few metrics per run. That answered "how fast is a fault caught". This notebook asks the questions that come after a QA system has been running for a while.

First, **do verdicts change?** A verdict issued in March was made with only the data up to March. Once more data arrives the same hour can be judged differently, because normals are better known, neighbours have reported and a wobble has turned out to be a lasting shift. If verdicts move, anyone who trained on last month's verdicts trained on a different dataset than anyone training today.

Second, **who watches the checks?** Every check fires at some background rate. When that rate jumps it means a new fault, unusual weather or a check that has quietly changed. A **control chart** draws a band around each rate from its own recent history and raises an alert when it leaves.

Third, **what happens to a human review?** Notebook 05 checked 30 alarms by hand. Those labels are worth keeping as data beside the verdicts, so they change what a user sees without erasing what the machine said.

Everything here runs on the real station data, not the planted copy.

In [ ]:
# Setup: identical in every notebook. Data lives in ../data, which git ignores.
import os
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))
os.makedirs(DATA_DIR, exist_ok=True)
print("Data directory:", DATA_DIR)

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from wxlab import stations as st, reference as ref, faults as fl, qa, monitoring as mon

plt.rcParams["figure.dpi"] = 80
REF_DIR, QA_DIR = Path(DATA_DIR) / "reference", Path(DATA_DIR) / "qa"

reference = ref.load(REF_DIR / "hrrr")
model_elev = pd.read_parquet(REF_DIR / "hrrr_model_elevation.parquet").set_index("station_id").model_elevation_m
hourly = st.load_standardized(Path(DATA_DIR) / "standardized" / "isd", years=range(2021, 2025))
matched = ref.match(hourly, reference, model_elev).dropna(subset=["hrrr_air_temperature"])
older = st.load_standardized(Path(DATA_DIR) / "standardized" / "isd", station_ids=matched.station_id.unique(), years=range(2015, 2021))
clim = qa.climatology(older)
points = pd.read_parquet(REF_DIR / "station_points.parquet")
dups = fl.duplicate_pairs(matched)
nbrs = qa.neighbours(points, dups)

final = pd.read_parquet(QA_DIR / "real_flags.parquet")       # one run over all four years (notebook 05)
print(f"{len(matched):,} station-hours at {matched.station_id.nunique()} stations")

## Step 1 · Run the checks month by month on real data

`qa.monitor` replays history. For each month it runs every check on all data up to the end of that month and keeps only that month's verdicts. That's what a user would have seen on the first day of the next month. It takes a while (48 runs over a growing table) so the result is cached.

In [ ]:
mon_path, monflags_path = QA_DIR / "monitoring_real.parquet", QA_DIR / "monitoring_flags_real.parquet"
if mon_path.exists():
    metrics, asof = pd.read_parquet(mon_path), pd.read_parquet(monflags_path)
else:
    metrics, asof = qa.monitor(matched, clim, nbrs)
    metrics.to_parquet(mon_path); asof.to_parquet(monflags_path)
metrics["month"] = pd.PeriodIndex(metrics["month"], freq="M")
print(len(metrics), "monthly runs")

## Step 2 · How often do verdicts change?

Join each hour's verdict as first issued with its verdict in hindsight (the single run over all four years). Rows are the first verdict and columns where it ended up.

In [ ]:
j = mon.verdict_churn(asof, final)
print(f"{(j['first'] != j['final']).mean():.2%} of station-hours changed verdict")
print(j.change.value_counts(normalize=True).round(4).to_dict())
mon.churn_table(j).round(3)

In [ ]:
# When in the record do changes happen? Early months have the least history behind them.
j["month"] = j.time.dt.tz_localize(None).dt.to_period("M")
rate = j.assign(changed=j["first"] != j["final"]).groupby("month").changed.mean()
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(rate.index.to_timestamp(), rate * 100, marker=".")
ax.set_ylabel("verdicts changed (%)"); ax.set_title("Share of each month's verdicts that changed in hindsight", loc="left", fontsize=10)

*Draft note: describe what the numbers show once run. Expect most churn early in 2021, when normals and the shift detectors had the least history, and churn concentrated in suspect ↔ pass rather than in quarantine. Name the biggest mover.*

The lesson for a pipeline is that a verdict is a function of the data *and* of when it was computed. Storing verdicts with the run that produced them (a version), instead of overwriting them, is what lets a model name the exact verdicts it trained on.

## Step 3 · A control chart for every check

Each monthly run recorded how often each check fired. A band from the previous 12 months (median ± 4 robust standard deviations) shows when a rate leaves its normal range. Using only earlier months means a month never judges itself.

In [ ]:
rate_cols = [c for c in metrics.columns if c.startswith("rate_")] + ["coverage", "stations_quarantined"]
cc = mon.control_chart(metrics, rate_cols, floor=1e-4)
alerts = cc[cc.alert]
print(len(alerts), "alerts over", metrics.month.nunique(), "months")
alerts.groupby("metric").month.apply(lambda m: ", ".join(str(x) for x in m)).to_frame("months")

In [ ]:
show = ["rate_reference", "rate_neighbour", "rate_timing", "coverage"]
fig, axes = plt.subplots(len(show), 1, figsize=(10, 8), sharex=True)
for ax, c in zip(axes, show):
    d = cc[cc.metric == c]; x = d.month.dt.to_timestamp()
    ax.fill_between(x, d.lower, d.upper, color="tab:blue", alpha=0.12, lw=0)
    ax.plot(x, d.value, marker=".", color="k", lw=0.8)
    ax.scatter(x[d.alert], d.value[d.alert], color="tab:red", zorder=3)
    ax.set_ylabel(c.replace("rate_", ""), fontsize=9)
axes[0].set_title("Monthly check rates with their control bands (red = alert)", loc="left", fontsize=10)

*Draft note: once run, go through the red points. Expect three kinds: storm months (coverage drops, neighbour and reference rates rise together across many stations), single-station faults (one station drives a rate up; check with the per-station breakdown), and the first months of 2021 when the band is still forming. Point out any alert that matches a known event from notebooks 03 and 05.*

An alert on a check rate is a different question from an alert on a station. It asks whether the QA system still behaves as it did. A check whose rate drifts slowly upward over a year may be fine, or may be reacting to something that changed upstream, like a new model version in the reference. That's why the rates themselves are worth monitoring.

## Step 4 · Reviews as data

The 30 alarms reviewed by hand in notebook 05 become override records: station, month, label, the action it implies, the reason, who reviewed and when. They're joined beside the verdicts as a separate column. The automatic verdict stays as it was, so it can still be scored, and a review can be undone by deleting a row.

In [ ]:
review = pd.read_csv(Path("..") / "docs" / "real-flag-review.csv", dtype={"station_id": str})
overrides = mon.overrides_from_review(review, reviewer="owner", reviewed_at="2026-11-08")
reviewed = mon.apply_overrides(final, overrides)
print(overrides.action.value_counts().to_dict())
q0, q1 = (reviewed.verdict == "quarantine").sum(), (reviewed.reviewed_verdict == "quarantine").sum()
print(f"quarantined hours: {q0:,} automatic → {q1:,} after review")
overrides.head()

*Draft note: report how many quarantined hours the 20 "weather" reviews lift, and that the "fault" reviews confirm the biggest offenders. Mention that the reviewed sample is random, so the reviewed verdict on unreviewed months is unchanged: review improves the reviewed rows, it doesn't fix the checks.*

The next step in a real system is to turn repeated review outcomes into rules. Notebook 05's false alarms fell into a few weather patterns (coastal heat waves in offshore wind, local winds HRRR times an hour off, fog). A rule that recognises one pattern changes the checks, which then needs the whole recall and precision evaluation again, on planted and real data. Reviews feed the rules. They don't replace the evaluation.

## What surprised me

*Draft: fill in after running. Candidates: how much churn there is, which check's rate is least stable, and whether any alert caught something the station-level checks missed.*

## Check yourself

<details><summary>Why compare a verdict "as first issued" with one "in hindsight" instead of just rerunning everything each month?</summary>
Rerunning everything is what a pipeline should do. The comparison measures what a user who trusted the earlier verdict actually got. If churn is high, training sets built at different times differ, and dataset versions are needed to reproduce a model.
</details>

<details><summary>Why build the control band from earlier months only?</summary>
If the current month were included it would widen its own band and could hide its own alert. Using only the past also matches what a live system knows when it decides whether to alert.
</details>

<details><summary>Why keep the automatic verdict when a person has overridden it?</summary>
So the checks can still be scored on what they actually said, the review can be audited or undone, and a later rule change can be compared with both.
</details>

New terms from this notebook are in [`docs/glossary.md`](../docs/glossary.md).